In [1]:
import os
base_path ='/Users/yifanli/Desktop/dataset/TumorSegmentation/BraTS20/MICCAI_BraTS2020_TrainingData'

# Initialize lists for each modality and labels
flair_paths = []
t1_paths = []
t1gd_paths = []
t2_paths = []
label_paths = []

# Initialize a dictionary to track label availability
label_availability = {}

# Loop through each patient folder
for patient_folder in os.listdir(base_path):
    patient_path = os.path.join(base_path, patient_folder)
    
    if os.path.isdir(patient_path):
        # Reset or declare label found flag for each patient
        label_found = False
        
        for file in os.listdir(patient_path):
            file_path = os.path.join(patient_path, file)

            if file.endswith('_flair.nii.gz'):
                flair_paths.append(file_path)
            elif file.endswith('_t1.nii.gz'):
                t1_paths.append(file_path)
            elif file.endswith('_t1ce.nii.gz'):
                t1gd_paths.append(file_path)
            elif file.endswith('_t2.nii.gz'):
                t2_paths.append(file_path)
            elif file.endswith('_seg.nii.gz'):
                label_paths.append(file_path)
                label_found = True
                label_availability[patient_folder] = 'ManuallyCorrected'

        # If manually corrected label is not found, look for GlistrBoost label
        if not label_found:
            for file in os.listdir(patient_path):
                if file.endswith('_GlistrBoost.nii.gz'):
                    print(patient_path)
                    file_path = os.path.join(patient_path, file)
                    label_paths.append(file_path)
                    label_availability[patient_folder] = 'GlistrBoost'

# Print the count of files in each list and availability of labels
print(f'FLAIR files: {len(flair_paths)}')
print(f'T1 files: {len(t1_paths)}')
print(f'T1Gd files: {len(t1gd_paths)}')
print(f'T2 files: {len(t2_paths)}')
print(f'Label files: {len(label_paths)}')

# Optionally, print the type of label used for each patient
# for patient, label_type in label_availability.items():
#     print(f'{patient}: {label_type}')

FLAIR files: 369
T1 files: 369
T1Gd files: 369
T2 files: 369
Label files: 369


In [2]:
import torch
from torch.utils.data import DataLoader
from dataset.dataloader import BrainDataset_WTonly
import tqdm
import os
import torch
import h5py
import numpy as np
import tqdm
import torch

In [3]:
# Function to optimize data types and save as HDF5
def save_to_hdf5(group, data):
    if isinstance(data, dict):
        for key, value in data.items():
            if isinstance(value, np.ndarray):
                # Save numpy arrays directly as datasets
                array = value
                if array.dtype == np.float64:
                    array = array.astype(np.float32)
                elif array.dtype == np.int64:
                    array = array.astype(np.int32)
                group.create_dataset(key, data=array, compression='gzip', compression_opts=9)
            elif isinstance(value, (np.number, float, int, str, bytes)):
                # Save basic data types as attributes
                group.attrs[key] = value
            else:
                # Recursively save nested structures
                subgroup = group.create_group(str(key))
                save_to_hdf5(subgroup, value)
    elif isinstance(data, (list, tuple)):
        # Handle lists and tuples
        for idx, item in enumerate(data):
            item_name = f'item_{idx}'
            if isinstance(item, np.ndarray):
                array = item
                if array.dtype == np.float64:
                    array = array.astype(np.float32)
                elif array.dtype == np.int64:
                    array = array.astype(np.int32)
                group.create_dataset(
                    item_name, data=array, compression='gzip', compression_opts=9
                )
            elif isinstance(item, (np.number, float, int, str, bytes)):
                group.attrs[item_name] = item
            else:
                subgroup = group.create_group(item_name)
                save_to_hdf5(subgroup, item)
    elif isinstance(data, np.ndarray):
        # Save numpy arrays
        array = data
        if array.dtype == np.float64:
            array = array.astype(np.float32)
        elif array.dtype == np.int64:
            array = array.astype(np.int32)
        group.create_dataset('data', data=array, compression='gzip', compression_opts=9)
    elif isinstance(data, (np.number, float, int, str, bytes)):
        # Save basic data types as attributes
        group.attrs['value'] = data
    else:
        # Convert unsupported types to string
        group.attrs['value'] = str(data)

def convert_pt_to_h5(pt_file_path, h5_file_path,only_cts = False):
    # Load the data from the .pt file
    data = torch.load(pt_file_path)
    if only_cts:
        if 'cts' in data:
            data = {'cts': data['cts']}
    with h5py.File(h5_file_path, 'w') as hf:
        save_to_hdf5(hf, data)
def convert_pt_to_h5_v2(data, h5_file_path,only_cts = False):
    if only_cts:
        if 'cts' in data:
            data = {'cts': data['cts']}
    with h5py.File(h5_file_path, 'w') as hf:
        save_to_hdf5(hf, data)

In [4]:
# Initialize your dataset
train_dataset = BrainDataset_WTonly(
    t1_paths=t1_paths,
    t1gd_paths=t1gd_paths,
    t2_paths=t2_paths,
    flair_paths=flair_paths,
    label_paths=label_paths,
    is_train=True
)

In [ ]:
preprocessed_data_dir ='/Users/yifanli/Desktop/dataset/AfterCT_BraTS20/preprocessed_data_BraTS20_WTonly'
if not os.path.exists(preprocessed_data_dir):
    os.makedirs(preprocessed_data_dir)

# New directory to save compressed .h5 files
h5_data_dir = '/Users/yifanli/Desktop/dataset/AfterCT_BraTS20/preprocessed_data_BraTS20_h5_WTonly'
if not os.path.exists(h5_data_dir):
    os.makedirs(h5_data_dir)
    

    
for idx in tqdm.tqdm(range(len(train_dataset))):
    
    data_filename = os.path.join(preprocessed_data_dir, f'data_{idx+1}.pt')
    data_filename_h5 = os.path.join(h5_data_dir, f'data_{idx+1}.h5')
#     if os.path.exists( data_filename):
#         print(f'Skipping, {data_filename_h5} already exists.')
#         continue  # Skip to the next file
    data = train_dataset[idx]
#     torch.save(data, data_filename)
    convert_pt_to_h5_v2(data, data_filename_h5 ,only_cts=False)

  0%|                                                   | 0/369 [00:00<?, ?it/s]

In [12]:
from utils.preprocessing_support import HDF5BrainDataset
# Initialize the dataset
previous_h5_data_dir = '/Users/yifanli/Desktop/dataset/AfterCT_BraTS20/preprocessed_data_BraTS20_h5_WTonly'


dataset = HDF5BrainDataset(previous_h5_data_dir)
for ele in dataset:
    break

# run PointEmbedding.py in GPU 
update the folder "preprocessed_data_BraTS20_WTonly"

In [13]:
from torch.utils.data import Dataset
def load_from_hdf5(group):
    """Recursively load all data from an HDF5 group into a Python dict."""
    result = {}
    # Load attributes at the current level
    for attr_key in group.attrs:
        result[attr_key] = group.attrs[attr_key]

    # For each key in this group
    for key in group.keys():
        item = group[key]
        if isinstance(item, h5py.Dataset):
            result[key] = item[:]
        elif isinstance(item, h5py.Group):
            # Recursively load sub-group
            result[key] = load_from_hdf5(item)
    return result


class HDF5BrainDataset(Dataset):
    def __init__(self, h5_data_dir):
        self.h5_data_dir = h5_data_dir
        self.data_files = sorted(
            [f for f in os.listdir(h5_data_dir) if f.endswith('.h5')],
            key=lambda x: int(os.path.splitext(x)[0].split('_')[1])
        )
        
    def __len__(self):
        return len(self.data_files)

    def __getitem__(self, index):
        """
        Reads from HDF5 and returns a Python dictionary with the data. 
        This is READ-ONLY: changes to the returned dict do not save back.
        """
        h5_file_name = self.data_files[index]
        h5_file_path = os.path.join(self.h5_data_dir, h5_file_name)

        with h5py.File(h5_file_path, 'r') as hf:
            # Adjust the keys you actually want to load
            data = load_from_hdf5(hf)
        return data


# Initialize the dataset

current_h5_data_dir =  '/Users/yifanli/Desktop/dataset/AfterCT_BraTS20/preprocessed_data_BraTS20_h5_WTonly'
pre_version_h5_data_dir =  'light_h5_data'

current_dataset = HDF5BrainDataset(current_h5_data_dir )
pre_version_dataset =HDF5BrainDataset(pre_version_h5_data_dir)


In [14]:
# Define the output folder for the light HDF5 files
h5_output_path = "light_h5_data_WTonly"
if not os.path.exists(h5_output_path):
    os.makedirs(h5_output_path)

# Loop over paired items from the previous and current datasets
for i, ( cur_data_dict,pre_light_dict) in tqdm.tqdm(enumerate(zip(current_dataset,pre_version_dataset))):
    
    for key in ['img_embedding']:
        if key not in cur_data_dict and key in pre_light_dict:
            cur_data_dict[key] = pre_light_dict[key]


    # --- Save the updated (light) version to a new HDF5 file ---
    out_file_name = current_dataset.data_files[i]
    out_file_path = os.path.join(h5_output_path, out_file_name)
    
    with h5py.File(out_file_path, 'w') as hf:
        save_to_hdf5(hf, cur_data_dict)


368it [59:13,  9.66s/it]


In [15]:
test_h5_data_dir =  'light_h5_data_WTonly'

test_dataset = HDF5BrainDataset(test_h5_data_dir)

In [16]:
for ele in test_dataset:
    print(ele.keys())
    print(ele['cts']['TC_l'].keys())
    break

In [22]:
ele['cts']['TC_s']['item_0'].shape

(200, 4096, 3)

In [36]:
# import h5py
# import numpy as np
# import os

# def preprocess_h5_to_final_shape(input_h5_path, output_h5_path):
#     # 1) Read from your original HDF5
#     with h5py.File(input_h5_path, 'r') as hf_in:
#         img_emb = hf_in['img_embedding'][...]  # shape (1024,4,4,4)
        
#         # Suppose you have subgroups like:
#         #  - hf_in['cts']['TC_s']['item_1'] = infiltration distances
#         #  - hf_in['cts']['TC_s']['item_2'] = contour embeddings
#         #   etc.

#         # 2) Do your indexing / flipping / concatenation just once
#         indices = np.concatenate([np.arange(i + 25, i + 50) for i in range(0, 200, 50)])
        
#         dis_tcs = hf_in['cts']['TC_s']['item_1'][indices]

#         g_emb_tcs = hf_in['cts']['TC_s']['item_2'][ indices]

#         g_emb_tcs = np.flip(g_emb_tcs, axis=1)  # flip dimension #1, if needed

#         dis_wtl = hf_in['cts']['WT_l']['item_1'][indices]
#         g_emb_wtl = hf_in['cts']['WT_l']['item_2'][indices]

#         dis_tcl = hf_in['cts']['TC_l']['item_1'][...]
#         g_emb_tcl = hf_in['cts']['TC_l']['item_2'][...]

#         all_scores = np.concatenate([dis_tcs, dis_tcl, dis_wtl], axis=0)
#         all_g_embs = np.concatenate([g_emb_tcs, g_emb_tcl, g_emb_wtl], axis=0)

#     # 3) Save these final shapes into a **new** HDF5 file
#     with h5py.File(output_h5_path, 'w') as hf_out:
#         hf_out.create_dataset('img_embedding', data=img_emb)
#         hf_out.create_dataset('contour_embedding', data=all_g_embs)
#         hf_out.create_dataset('contour_scores', data=all_scores)

#     print(f"Saved preprocessed data to {output_h5_path}")

# if __name__ == "__main__":
#     original_dir = "light_h5_data_WTonly"
#     output_dir = "light_h5_data_preprocessed"
#     os.makedirs(output_dir, exist_ok=True)

#     for fname in os.listdir(original_dir):
#         if fname.endswith(".h5"):
#             input_path = os.path.join(original_dir, fname)
#             output_path = os.path.join(output_dir, fname)
#             preprocess_h5_to_final_shape(input_path, output_path)


Saved preprocessed data to light_h5_data_preprocessed/data_2.h5
Saved preprocessed data to light_h5_data_preprocessed/data_292.h5
Saved preprocessed data to light_h5_data_preprocessed/data_103.h5
Saved preprocessed data to light_h5_data_preprocessed/data_70.h5
Saved preprocessed data to light_h5_data_preprocessed/data_152.h5
Saved preprocessed data to light_h5_data_preprocessed/data_21.h5
Saved preprocessed data to light_h5_data_preprocessed/data_230.h5
Saved preprocessed data to light_h5_data_preprocessed/data_261.h5
Saved preprocessed data to light_h5_data_preprocessed/data_321.h5
Saved preprocessed data to light_h5_data_preprocessed/data_83.h5
Saved preprocessed data to light_h5_data_preprocessed/data_15.h5
Saved preprocessed data to light_h5_data_preprocessed/data_166.h5
Saved preprocessed data to light_h5_data_preprocessed/data_44.h5
Saved preprocessed data to light_h5_data_preprocessed/data_137.h5
Saved preprocessed data to light_h5_data_preprocessed/data_315.h5
Saved preprocesse

Saved preprocessed data to light_h5_data_preprocessed/data_345.h5
Saved preprocessed data to light_h5_data_preprocessed/data_194.h5
Saved preprocessed data to light_h5_data_preprocessed/data_205.h5
Saved preprocessed data to light_h5_data_preprocessed/data_215.h5
Saved preprocessed data to light_h5_data_preprocessed/data_184.h5
Saved preprocessed data to light_h5_data_preprocessed/data_355.h5
Saved preprocessed data to light_h5_data_preprocessed/data_244.h5
Saved preprocessed data to light_h5_data_preprocessed/data_304.h5
Saved preprocessed data to light_h5_data_preprocessed/data_126.h5
Saved preprocessed data to light_h5_data_preprocessed/data_55.h5
Saved preprocessed data to light_h5_data_preprocessed/data_177.h5
Saved preprocessed data to light_h5_data_preprocessed/data_92.h5
Saved preprocessed data to light_h5_data_preprocessed/data_270.h5
Saved preprocessed data to light_h5_data_preprocessed/data_361.h5
Saved preprocessed data to light_h5_data_preprocessed/data_221.h5
Saved prepro

Saved preprocessed data to light_h5_data_preprocessed/data_170.h5
Saved preprocessed data to light_h5_data_preprocessed/data_212.h5
Saved preprocessed data to light_h5_data_preprocessed/data_183.h5
Saved preprocessed data to light_h5_data_preprocessed/data_352.h5
Saved preprocessed data to light_h5_data_preprocessed/data_243.h5
Saved preprocessed data to light_h5_data_preprocessed/data_303.h5
Saved preprocessed data to light_h5_data_preprocessed/data_263.h5
Saved preprocessed data to light_h5_data_preprocessed/data_323.h5
Saved preprocessed data to light_h5_data_preprocessed/data_81.h5
Saved preprocessed data to light_h5_data_preprocessed/data_232.h5
Saved preprocessed data to light_h5_data_preprocessed/data_150.h5
Saved preprocessed data to light_h5_data_preprocessed/data_23.h5
Saved preprocessed data to light_h5_data_preprocessed/data_290.h5
Saved preprocessed data to light_h5_data_preprocessed/data_101.h5
Saved preprocessed data to light_h5_data_preprocessed/data_72.h5
Saved preproc